In [1]:
# ============================================================
# FIX FAISS / OPENMP CRASH
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print(
    "OpenMP Fix Applied"
)

OpenMP Fix Applied


In [2]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import pandas as pd

import pickle

from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings

from langchain_openai import ChatOpenAI

from langchain_community.vectorstores import FAISS

from rank_bm25 import BM25Okapi

from duckduckgo_search import DDGS

print(
    "Libraries Loaded Successfully"
)

Libraries Loaded Successfully


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/2441754296.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

print(
    "Environment Variables Loaded"
)

Environment Variables Loaded


In [5]:
# ============================================================
# LOAD EVALUATION DATASET
# ============================================================

eval_df = pd.read_csv(
    "../notebooks/evaluation_dataset.csv"
)

print(
    "Rows:",
    len(eval_df)
)

eval_df.head()

Rows: 1000


,query,article,reference_summary
0,solar observatory aims to provide better under...,cnn nasa has postponed for one day the schedul...,solar observatory aims to provide better under...
1,mary j bliges new album my life ii,cnn seventeen years after the release of her b...,mary j bliges new album my life ii is a follow...
2,germany coach joachim low claims spain are fav...,cnn germany coach joachim low has tried to lif...,germany coach joachim low claims spain are fav...
3,presidential historians weigh in on how histor...,cnn with record low approval ratings and inten...,presidential historians weigh in on how histor...
4,chinas gold medal gymnasts cleared of competin...,cnn chinas olympic gold medal gymnasts have be...,chinas gold medal gymnasts cleared of competin...


In [6]:
# ============================================================
# LOAD OPENAI EMBEDDINGS
# ============================================================

embeddings = OpenAIEmbeddings(

    model="text-embedding-3-small"

)

print(
    "Embeddings Loaded Successfully"
)

Embeddings Loaded Successfully


In [7]:
# ============================================================
# LOAD FAISS
# ============================================================

vector_store = FAISS.load_local(

    "../notebooks/faiss_20k",

    embeddings,

    allow_dangerous_deserialization=True

)

print(
    "FAISS Loaded Successfully"
)

FAISS Loaded Successfully


In [8]:
# ============================================================
# LOAD CHUNKS
# ============================================================

with open(

    "../notebooks/all_chunks.pkl",

    "rb"

) as f:

    all_chunks = pickle.load(
        f
    )

print(
    "Chunks Loaded:",
    len(all_chunks)
)

Chunks Loaded: 160571


In [9]:
# ============================================================
# TOKENIZE CHUNKS
# ============================================================

tokenized_chunks = [

    chunk.lower().split()

    for chunk in all_chunks

]

print(
    "Tokenization Completed"
)

Tokenization Completed


In [10]:
# ============================================================
# LOAD BM25
# ============================================================

bm25 = BM25Okapi(
    tokenized_chunks
)

print(
    "BM25 Loaded Successfully"
)

BM25 Loaded Successfully


In [11]:
# ============================================================
# WEB SEARCH FUNCTION
# ============================================================

def web_search(

    query,

    max_results=5

):

    results = []

    with DDGS() as ddgs:

        search_results = list(

            ddgs.text(

                query,

                max_results=max_results

            )

        )

    for item in search_results:

        results.append(

            item["body"]

        )

    return results

In [12]:
# ============================================================
# LOAD GPT4o
# ============================================================

llm = ChatOpenAI(

    model="gpt-4o",

    temperature=0

)

print(
    "GPT4o Loaded Successfully"
)

GPT4o Loaded Successfully


In [13]:
# ============================================================
# RESET RESULTS
# ============================================================

results = []

print(
    "Results Reset Successfully"
)

Results Reset Successfully


In [14]:
# ============================================================
# FUSION GPT4o EVALUATION PIPELINE
# ============================================================

for idx, row in eval_df.iterrows():

    if idx >= 500:
        break

    if idx % 50 == 0:

        print(
            f"Processing Row {idx}"
        )

    query = str(
        row["query"]
    )

    reference_summary = str(
        row["reference_summary"]
    )

    # ========================================================
    # DENSE RETRIEVAL
    # ========================================================

    dense_docs = vector_store.similarity_search(

        query,

        k=5

    )

    dense_texts = [

        doc.page_content

        for doc in dense_docs

    ]

    # ========================================================
    # SPARSE RETRIEVAL
    # ========================================================

    tokenized_query = query.lower().split()

    sparse_indices = bm25.get_top_n(

        tokenized_query,

        list(
            range(
                len(all_chunks)
            )
        ),

        n=5

    )

    sparse_docs = [

        all_chunks[idx]

        for idx in sparse_indices

    ]

    # ========================================================
    # WEB SEARCH
    # ========================================================

    web_docs = web_search(

        query,

        max_results=5

    )

    # ========================================================
    # FUSION
    # ========================================================

    fusion_docs = (

        dense_texts

        +

        sparse_docs

        +

        web_docs

    )

    # ========================================================
    # CONTEXT
    # ========================================================

    context = "\n\n".join(
        fusion_docs
    )

    # ========================================================
    # PROMPT
    # ========================================================

    prompt = f"""

    You are a helpful AI assistant.

    Use ONLY the provided context.

    Context:

    {context}

    Question:

    {query}

    Answer:

    """

    # ========================================================
    # GENERATE RESPONSE
    # ========================================================

    response = llm.invoke(
        prompt
    )

    answer = response.content

    # ========================================================
    # STORE RESULTS
    # ========================================================

    results.append({

        "pipeline":
        "Fusion GPT4o",

        "model":
        "GPT4o",

        "retrieval_method":
        "Fusion",

        "query":
        query,

        "reference_summary":
        reference_summary,

        "generated_answer":
        answer

    })

print(
    "\nFusion GPT4o Evaluation Completed"
)

Processing Row 0


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been

Processing Row 50


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been

Processing Row 100


...

Processing Row 400


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been

Processing Row 450


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1839/812884497.py:15: RuntimeWarning: This package (`duckduckgo_search`) has been

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [15]:
# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)

print(
    "Total Records:",
    len(results_df)
)

results_df.head()

Total Records: 493


,pipeline,model,retrieval_method,query,reference_summary,generated_answer
0,Fusion GPT4o,GPT4o,Fusion,solar observatory aims to provide better under...,solar observatory aims to provide better under...,The solar observatory aims to provide a better...
1,Fusion GPT4o,GPT4o,Fusion,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"Mary J. Blige's new album ""My Life II: The Jou..."
2,Fusion GPT4o,GPT4o,Fusion,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,Germany coach Joachim Low claims that Spain ar...
3,Fusion GPT4o,GPT4o,Fusion,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,Presidential historians suggest that history w...
4,Fusion GPT4o,GPT4o,Fusion,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,China's gold medal gymnasts were cleared of co...


In [16]:
# ============================================================
# MANUAL VALIDATION
# ============================================================

results_df[

    [

        "query",

        "reference_summary",

        "generated_answer"

    ]

].head(10)

,query,reference_summary,generated_answer
0,solar observatory aims to provide better under...,solar observatory aims to provide better under...,The solar observatory aims to provide a better...
1,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"Mary J. Blige's new album ""My Life II: The Jou..."
2,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,Germany coach Joachim Low claims that Spain ar...
3,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,Presidential historians suggest that history w...
4,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,China's gold medal gymnasts were cleared of co...
5,critics and viewers see stewart as victor after,critics and viewers see stewart as victor afte...,critics and viewers see Stewart as the victor ...
6,bahr idriss abu garda faces charges in deaths,bahr idriss abu garda faces charges in deaths ...,Bahr Idriss Abu Garda faces charges related to...
7,south africa beat cohosts india in world cup,south africa beat cohosts india in world cup c...,South Africa beat cohosts India in the World C...
8,rudy ruiz its become unfashionable to have an,rudy ruiz its become unfashionable to have an ...,Rudy Ruiz suggests that it has become unfashio...
9,fabio cannavaro is to join the italian national,fabio cannavaro is to join the italian nationa...,Fabio Cannavaro is to join the Italian nationa...


In [17]:
# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(

    "fusion_gpt4o_eval.csv",

    index=False

)

print(
    "Fusion GPT4o Evaluation Results Saved Successfully"
)

Fusion GPT4o Evaluation Results Saved Successfully


In [18]:
# ============================================================
# VERIFY SAVED FILE
# ============================================================

saved_df = pd.read_csv(
    "fusion_gpt4o_eval.csv"
)

print(
    saved_df.shape
)

saved_df.head()

(493, 6)


,pipeline,model,retrieval_method,query,reference_summary,generated_answer
0,Fusion GPT4o,GPT4o,Fusion,solar observatory aims to provide better under...,solar observatory aims to provide better under...,The solar observatory aims to provide a better...
1,Fusion GPT4o,GPT4o,Fusion,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"Mary J. Blige's new album ""My Life II: The Jou..."
2,Fusion GPT4o,GPT4o,Fusion,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,Germany coach Joachim Low claims that Spain ar...
3,Fusion GPT4o,GPT4o,Fusion,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,Presidential historians suggest that history w...
4,Fusion GPT4o,GPT4o,Fusion,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,China's gold medal gymnasts were cleared of co...


In [19]:
print(len(results))

493


In [20]:
print(results[-1]["query"])

the alshabaab militia seized the cities of bulo


In [21]:
print(len(results))

493


In [22]:
results

[{'pipeline': 'Fusion GPT4o',
  'model': 'GPT4o',
  'retrieval_method': 'Fusion',
  'query': 'solar observatory aims to provide better understanding of',
  'reference_summary': 'solar observatory aims to provide better understanding of suns role in space weather observatory to deliver images with resolution 10 times better than highdef tv fiveyear mission will examine suns magnetic field and violent solar events nasa observatory will snap image of sun in eight wavelengths every 10 seconds',
  'generated_answer': 'The solar observatory aims to provide a better understanding of the sun and its role in space weather events, such as solar flares, which can wreak havoc on Earth.'},
 {'pipeline': 'Fusion GPT4o',
  'model': 'GPT4o',
  'retrieval_method': 'Fusion',
  'query': 'mary j bliges new album my life ii',
  'reference_summary': 'mary j bliges new album my life ii is a followup to an album from 17 years ago the singer says that recently she has learned to say you know what i am deservin

In [23]:
print(len(eval_df))

1000


In [24]:
print(eval_df.shape)

print(len(eval_df))

eval_df.head()

(1000, 3)
1000


,query,article,reference_summary
0,solar observatory aims to provide better under...,cnn nasa has postponed for one day the schedul...,solar observatory aims to provide better under...
1,mary j bliges new album my life ii,cnn seventeen years after the release of her b...,mary j bliges new album my life ii is a follow...
2,germany coach joachim low claims spain are fav...,cnn germany coach joachim low has tried to lif...,germany coach joachim low claims spain are fav...
3,presidential historians weigh in on how histor...,cnn with record low approval ratings and inten...,presidential historians weigh in on how histor...
4,chinas gold medal gymnasts cleared of competin...,cnn chinas olympic gold medal gymnasts have be...,chinas gold medal gymnasts cleared of competin...


In [26]:
embeddings.embed_query(
    "test"
)

[-0.00986480712890625,
 0.0015592575073242188,
 0.0156707763671875,
 -0.0548095703125,
 -0.00640869140625,
 -0.012908935546875,
 0.00966644287109375,
 -0.01355743408203125,
 0.0286712646484375,
 0.007843017578125,
 0.03179931640625,
 -0.0065765380859375,
 0.0038814544677734375,
 0.01012420654296875,
 0.0140533447265625,
 0.044677734375,
 -0.059783935546875,
 -0.002445220947265625,
 -0.0511474609375,
 0.036529541015625,
 0.036163330078125,
 0.02435302734375,
 0.03997802734375,
 -0.043609619140625,
 0.034454345703125,
 -0.0195770263671875,
 -0.01401519775390625,
 0.0106353759765625,
 0.03118896484375,
 -0.041534423828125,
 0.05877685546875,
 -0.02886962890625,
 -0.0013904571533203125,
 -0.038818359375,
 0.055206298828125,
 0.004138946533203125,
 0.0236053466796875,
 0.018890380859375,
 -0.0056915283203125,
 -0.003925323486328125,
 -0.041473388671875,
 -0.044921875,
 0.0113677978515625,
 0.0272064208984375,
 0.0299835205078125,
 -0.0186767578125,
 -0.03436279296875,
 -0.038665771484375,
 